# Agent Capsules

Same scenarios, two modes (baseline vs capsule), side-by-side token + quality. Cell-by-cell version of `run.py`.

In [ ]:
import json, os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from _common import get_provider, score_task
from capsule import run_pipeline
from mock_pipeline import MockPipelineProvider
from rubric import DUE_DILIGENCE

pipeline = get_provider() if (os.getenv('ANTHROPIC_API_KEY') or os.getenv('OPENAI_API_KEY')) else MockPipelineProvider()
judge = get_provider()
print('pipeline:', pipeline.name, '   judge:', judge.name)

In [ ]:
scenarios = json.loads(Path('scenarios.json').read_text())['scenarios']
len(scenarios), [s['id'] for s in scenarios]

In [ ]:
rows = []
for sc in scenarios:
    base = run_pipeline(pipeline, sc, mode='baseline')
    caps = run_pipeline(pipeline, sc, mode='capsule')
    qb = score_task(provider=judge, task=sc['question'], candidate=base['summary'],
                    gold=sc['gold_summary'], rubric=DUE_DILIGENCE, task_id=sc['id']).aggregate()
    qc = score_task(provider=judge, task=sc['question'], candidate=caps['summary'],
                    gold=sc['gold_summary'], rubric=DUE_DILIGENCE, task_id=sc['id']).aggregate()
    rows.append({'id': sc['id'], 'tok_base': base['tokens'], 'tok_caps': caps['tokens'],
                 'esc': caps['escalations'], 'q_base': qb, 'q_caps': qc})
rows

In [ ]:
print(f"{'scenario':<20} {'base_tok':>9} {'caps_tok':>9} {'Δtok%':>7} {'esc':>4} {'q_base':>7} {'q_caps':>7} {'Δq':>6}")
print('-' * 80)
sb = sc_t = 0
qb_t = qc_t = 0.0
for r in rows:
    dt = (r['tok_caps'] - r['tok_base']) / r['tok_base'] * 100
    print(f"{r['id']:<20} {r['tok_base']:>9} {r['tok_caps']:>9} {dt:>+6.1f}% {r['esc']:>4} {r['q_base']:>7.2f} {r['q_caps']:>7.2f} {r['q_caps']-r['q_base']:>+6.2f}")
    sb += r['tok_base']; sc_t += r['tok_caps']; qb_t += r['q_base']; qc_t += r['q_caps']
n = len(rows)
print('-' * 80)
print(f"{'TOTAL':<20} {sb:>9} {sc_t:>9} {(sc_t-sb)/sb*100:>+6.1f}% {'':>4} {qb_t/n:>7.2f} {qc_t/n:>7.2f} {(qc_t-qb_t)/n:>+6.2f}")